<a href="https://colab.research.google.com/github/nibaskumar93n-debug/Morphoinformatics/blob/main/scRNA-seq_to_count_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -q https://cf.10xgenomics.com/releases/cell-exp/cellranger-7.1.0.tar.gz
!tar -xzf cellranger-7.1.0.tar.gz
!rm cellranger-7.1.0.tar.gz

import os
os.environ['PATH'] = '/content/cellranger-7.1.0:' + os.environ['PATH']

!cellranger --version

In [ ]:
# Just define sample names matching your downloaded file names
ena_samples = {
    'sample1': 'SRR0000001',  # replace with actual SRR numbers
    'sample2': 'SRR0000002',
    'sample3': 'SRR0000003',
}

sample_list = list(ena_samples.keys())
print("Samples defined:", sample_list)

In [ ]:
# Cell 1 — Upload your wget script
from google.colab import files
uploaded = files.upload()  # upload your wget script here

# Cell 2 — Make executable and run
!chmod +x *.sh
!bash *.sh

In [ ]:
import os
os.makedirs('/content/fastq_renamed', exist_ok=True)

sample_list = list(ena_samples.keys())

for i, sample in enumerate(sample_list, 1):
    r1_old = f'/content/fastq_files/{sample}_1.fastq.gz'
    r2_old = f'/content/fastq_files/{sample}_2.fastq.gz'

    r1_new = f'/content/fastq_renamed/{sample}_S{i}_L001_R1_001.fastq.gz'
    r2_new = f'/content/fastq_renamed/{sample}_S{i}_L001_R2_001.fastq.gz'

    os.rename(r1_old, r1_new)
    os.rename(r2_old, r2_new)
    print(f"✓ {sample} renamed")

In [ ]:
print("Downloading human reference genome...")
print("This will take 15-20 minutes...")

!wget -q https://cf.10xgenomics.com/supp/cell-exp/refdata-gex-GRCh38-2020-A.tar.gz
!tar -xzf refdata-gex-GRCh38-2020-A.tar.gz
!rm refdata-gex-GRCh38-2020-A.tar.gz

print("✓ Reference genome ready!")

In [ ]:
sample_list = list(ena_samples.keys())

for i, sample in enumerate(sample_list, 1):
    print(f"\nRunning Cell Ranger for {sample}...")
    !cellranger count \
        --id={sample}_output \
        --transcriptome=/content/refdata-gex-GRCh38-2020-A \
        --fastqs=/content/fastq_renamed \
        --sample={sample} \
        --localcores=4 \
        --localmem=16 \
        --nosecondary
    print(f"✓ {sample} done!")

In [ ]:
for sample in sample_list:
    print(f"\nOutput files for {sample}:")
    !ls /content/{sample}_output/outs/filtered_feature_bc_matrix/

In [ ]:
from google.colab import files
import shutil

for sample in sample_list:
    matrix_dir = f'/content/{sample}_output/outs/filtered_feature_bc_matrix'
    zip_path = f'/content/{sample}_matrix'

    shutil.make_archive(zip_path, 'zip', matrix_dir)
    files.download(f'{zip_path}.zip')
    print(f"✓ {sample} matrix downloaded!")

print("\n✓ All done!")